###1. Basic Tasks

**1. Create a serverless SQL warehouse and run 3 exploratory queries against a gold table from Day 6/8.**

In [0]:
%sql
-- Top 10 best-selling products
SELECT
    product_rank,
    product_id,
    product_name,
    category,
    total_quantity_sold,
    total_revenue
FROM dev.gold.products_best_selling
ORDER BY product_rank
LIMIT 10;

In [0]:
%sql
-- Revenue by category
SELECT
    category,
    SUM(total_quantity_sold) AS total_quantity_sold,
    SUM(total_revenue) AS total_revenue
FROM dev.gold.products_best_selling
GROUP BY category
ORDER BY total_revenue DESC;

In [0]:
%sql
-- Highest-revenue products
SELECT
    product_name,
    category,
    total_quantity_sold,
    total_revenue
FROM dev.gold.products_best_selling
ORDER BY total_revenue DESC
LIMIT 10;

**2. Build an AI/BI dashboard with at least 2 visualizations sourced from those queries.**

![dashboard1_1789408966572.png](./dashboard1_1789408966572.png "dashboard1_1789408966572.png")

**3. Publish the dashboard and test the 'Ask Genie' feature with 2 plain-English questions.**

After creating the above dashboard I have created the asked these two below questions:
1. Give me the Highest selling product, as per total revenue
```text
Genie Ans: 
Highest Selling Product by Total Revenue
Sint Asperiores is the highest selling product with $755,727 in total revenue. This Furniture category product sold 354 units.
```
2. Give me the category with the least revenue
```text
Genie Ans: 
Category with Least Revenue
Home is the category with the least revenue at $19,849,308, with 9,729 units sold.
```

###2. Intermediate Tasks

**4. Add a filter to your dashboard (e.g., by region or date range) and confirm both visualizations respond to it.**

![dashboard_filter1_1789409034378.png](./dashboard_filter1_1789409034378.png "dashboard_filter1_1789409034378.png")
![dashboard_filter2_1789409045445.png](./dashboard_filter2_1789409045445.png "dashboard_filter2_1789409045445.png")

**5. Set up a Genie Agent scoped to your sales tables: write at least 2 instructions and 2 sample queries to steer its behavior.**

**1. Source file add**
- added this source file `dev.silver.sales_cleaned` in the Genie Agents

**2. Instructions add**

I have added these two instructions
- When calculating sales revenue, use the total_amount column. Do not use discount_amount as revenue. For total sales, calculate SUM(total_amount).

- When a user asks about sales for a specific date, month, or year, use order_date for date filtering. Always apply the requested date filter before calculating sales metrics.

**3. Sample query added**
- Monthly sales revenue
```sql
SELECT
    DATE_TRUNC('month', order_date) AS month,
    SUM(total_amount) AS total_revenue
FROM dev.silver.sales_cleaned
GROUP BY DATE_TRUNC('month', order_date)
ORDER BY month;
```
- Top 10 customers by revenue
```sql
SELECT
    customer_id,
    SUM(total_amount) AS total_revenue
FROM dev.silver.sales_cleaned
GROUP BY customer_id
ORDER BY total_revenue DESC
LIMIT 10;
```
![query1_1789409164466.png](./query1_1789409164466.png "query1_1789409164466.png")
![query2_1789409172024.png](./query2_1789409172024.png "query2_1789409172024.png")

**6. Ask your Genie Agent a question it initially answers incorrectly or vaguely, then improve its
instructions/sample queries/trusted assets until it answers correctly — document the before and
after.**

###3. Advanced Tasks

**7. Design a Genie Agent curation checklist for Cyntexa: what Unity Catalog metadata (column
comments, table descriptions) needs to exist before a Genie Agent can be trusted for
executive-facing questions.**

**Genie Agent Curation Checklist — Cyntexa**

Before trusting a Genie Agent for executive-facing questions, ensure:

1. **Table description** — Clearly explain the table's purpose and what one row represents.
2. **Column comments** — Add business-friendly descriptions for important columns.
3. **Business metric definitions** — Clearly define metrics such as:

   * Sales = `SUM(total_amount)`
   * Orders = `COUNT(DISTINCT order_id)`
   * Customers = `COUNT(DISTINCT customer_id)`
4. **Data grain** — Document whether each row represents an order, transaction, product, etc.
5. **Date definition** — Identify the correct date column for time-based analysis.
6. **Data quality** — Check for NULLs, duplicates, invalid values, and incorrect records.
7. **Genie instructions** — Add explicit business rules for calculations and terminology.
8. **Trusted assets/queries** — Create validated SQL for critical executive KPIs.
9. **Unity Catalog governance** — Verify permissions and ensure sensitive data isn't unnecessarily exposed.
10. **Validation** — Compare Genie answers against manually validated SQL results before using it for executive reporting.

**Minimum requirement:** Clear Unity Catalog metadata + defined business metrics + validated data + trusted queries + proper governance.


**8. Compare Photon, Predictive I/O, and Intelligent Workload Management's roles in why a serverless
warehouse can answer ad hoc dashboard queries fast, and use that to justify serverless over
classic/pro warehouses for this use case.**

**Photon vs Predictive I/O vs Intelligent Workload Management**

| Feature                                   | Role                                                                                                                                              |
| ----------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Photon**                                | Databricks' high-performance query engine. It accelerates SQL operations such as scans, joins, aggregations, and filtering.                       |
| **Predictive I/O**                        | Optimizes data access by intelligently predicting which data needs to be read, reducing unnecessary I/O and improving query performance.          |
| **Intelligent Workload Management (IWM)** | Automatically manages compute resources and workload concurrency so multiple ad-hoc queries can run efficiently without manually tuning clusters. |

**Why Serverless is better for this use case**

For **ad-hoc dashboard queries**, users need fast response times without managing infrastructure.

**Serverless SQL Warehouse:**

```text
Dashboard
    ↓
Serverless SQL Warehouse
    ↓
Photon + Predictive I/O + Intelligent Workload Management
    ↓
Fast query results
```

Serverless is preferable because Databricks manages **compute provisioning, scaling, concurrency, and optimization automatically**.

With **classic/pro warehouses**, you have more infrastructure and configuration responsibility, such as cluster sizing and scaling decisions. They can still provide excellent performance, but are less convenient for unpredictable, interactive workloads.

So for Cyntexa's executive dashboards and ad-hoc SQL analysis, **Serverless SQL Warehouse is the better choice** because Photon accelerates query execution, Predictive I/O reduces unnecessary data reads, and Intelligent Workload Management handles changing workloads and concurrency automatically. This provides fast, low-management analytics compared with managing classic/pro warehouse infrastructure.


**9. (Data Analyst-led) Build a 3-visualization executive dashboard answering a real business question (e.g., 'are we hitting our quarterly revenue target by region?'), publish it, and write the 2-3 sentence 'Ask Genie' instructions you'd give a VP who has never seen the underlying tables.**

![dashboard_page1_1789409347940.png](./dashboard_page1_1789409347940.png "dashboard_page1_1789409347940.png")
![dashboard_page2_1789409355997.png](./dashboard_page2_1789409355997.png "dashboard_page2_1789409355997.png")
![dashboard_page3_1789409365377.png](./dashboard_page3_1789409365377.png "dashboard_page3_1789409365377.png")
This is the "Ask Genie" instruction I would give to VP: 
- Use the Sales Performance page to understand overall revenue, sales trends, and performance by region or segment. Use the Customer Analytics page to identify customer growth, retention, and high-value customer segments. Finally, use the Product Insights page to determine which products are driving revenue and which products or categories may need attention, and summarize the key business opportunities or risks.